In [ ]:
import json
import re
import unicodedata
from pathlib import Path

# Carpeta desde la que ejecutas el notebook: develop
BASE_DIR = Path.cwd()

# Raíz del proyecto
PROJECT_ROOT = BASE_DIR.parent

# GeoJSON original en config
INPUT_GEOJSON = PROJECT_ROOT / "config" / "provincias_es_azuremaps_final.geojson"

# Carpeta de salida dentro de config
OUTPUT_DIR = PROJECT_ROOT / "config" / "geojson_por_ccaa"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def normalizar_nombre(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = texto.lower().strip()
    texto = texto.replace("-", "_").replace("/", "_")
    texto = re.sub(r"\s+", "_", texto)
    texto = re.sub(r"[^a-z0-9_]", "", texto)
    texto = re.sub(r"_+", "_", texto)
    return texto.strip("_")

# Cargar GeoJSON original
with INPUT_GEOJSON.open("r", encoding="utf-8") as f:
    geojson = json.load(f)

features = geojson.get("features", [])

# Agrupar por CCAA
grupos = {}
for feature in features:
    props = feature.get("properties", {})
    ccaa = props.get("CCAA")
    if not ccaa:
        continue
    grupos.setdefault(ccaa, []).append(feature)

# Crear un GeoJSON por cada CCAA
for ccaa, features_ccaa in grupos.items():
    codigo_ccaa = features_ccaa[0].get("properties", {}).get("Codigo_CCAA", "")
    nombre_normalizado = normalizar_nombre(ccaa)
    nombre_fichero = f"provincias_{nombre_normalizado}.geojson"

    geojson_salida = {
        "type": "FeatureCollection",
        "name": f"provincias_{nombre_normalizado}",
        "crs": geojson.get("crs"),
        "features": features_ccaa
    }

    output_path = OUTPUT_DIR / nombre_fichero
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(geojson_salida, f, ensure_ascii=False)

    print(
        f"Creado: {output_path} | "
        f"CCAA={ccaa} | Código={codigo_ccaa} | Provincias={len(features_ccaa)}"
    )

print(f"\nTotal de archivos generados: {len(grupos)}")
print(f"Carpeta de salida: {OUTPUT_DIR}")